# LeetCode 3028 — Ant on the Boundary bằng Python

Link đề: https://leetcode.com/problems/ant-on-the-boundary/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Prefix Sum

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def returnToBoundaryCount(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một con kiến xuất phát **trên đường biên**; mỗi phần tử `nums[i]` bảo kiến đi `|nums[i]|` bước sang **trái** (nếu âm) hoặc **phải** (nếu dương); đếm số lần kiến **đang đứng đúng trên biên** **sau khi đi xong** một đoạn.
2. Hiểu vì sao bài chính là **tổng tiền tố (prefix sum)**: vị trí sau `i + 1` đoạn là `nums[0] + ... + nums[i]`, và ta đếm có **bao nhiêu tổng tiền tố bằng `0`**.
3. Phân biệt **"kiến đi qua biên giữa chừng"** (không tính) với **"kiến dừng đúng trên biên sau khi hết đoạn đi"** (có tính).
4. Đi từ cách **mô phỏng từng bước một** → **cộng dồn vị trí `pos`** → **`itertools.accumulate`** → **`Counter` của các tổng tiền tố**.
5. Tránh các lỗi kinh điển: đếm cả lúc đi **qua biên giữa chừng**, đếm **vị trí xuất phát**, nhầm **đổi dấu** với **chạm biên**, và quên cộng dồn (`pos = nums[i]` thay vì `pos += nums[i]`).

## 2. Đề bài & ví dụ

Một con kiến đứng trên một **sợi dây** (đường thẳng) và **ban đầu nó ở đúng vị trí của đường biên**. Bạn được cho một mảng số nguyên khác `0` tên `nums`. Kiến di chuyển lần lượt theo từng phần tử `nums[i]`, **bắt đầu từ `i = 0`**:

- nếu `nums[i] < 0`, nó đi sang **trái** `|nums[i]|` đơn vị;
- nếu `nums[i] > 0`, nó đi sang **phải** `nums[i]` đơn vị.

Hãy trả về **số lần kiến quay lại đường biên**.

**Lưu ý:**

- Mỗi lần đi, kiến di chuyển **trọn** `|nums[i]|` đơn vị. Ta **chỉ kiểm tra** xem kiến có đang ở biên **sau khi đã đi xong** `|nums[i]|` đơn vị đó hay không.
- Nếu kiến **chỉ đi ngang qua** biên **trong lúc** đang đi (không dừng ở đó khi đi xong đoạn) thì **không** tính là quay lại biên.

**Ví dụ 1**

- Input: `nums = [2, 3, -5]`
- Output: `1`
- Giải thích: sau bước 1 kiến ở vị trí `2` (cách biên `2` về bên phải); sau bước 2 ở `5`; sau bước 3 ở `0`, đúng biên. Vậy `1` lần.

**Ví dụ 2**

- Input: `nums = [3, 2, -3, -4]`
- Output: `0`
- Giải thích: các vị trí sau từng bước là `3, 5, 2, -2`. Kiến **đi qua** biên giữa chừng ở bước cuối (từ `2` sang `-2`) nhưng không dừng ở đó nên không tính. Vậy `0` lần.

**Ràng buộc:** `1 <= nums.length <= 100`; `-10 <= nums[i] <= 10`; `nums[i] != 0`.

In [1]:
from typing import List
from itertools import accumulate
from collections import Counter
import random

examples = [
    ([2, 3, -5], 1),
    ([3, 2, -3, -4], 0),
    ([-3, 3], 1),
    ([1, -1, 1, -1], 2),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [2, 3, -5] -> kỳ vọng 1
nums = [3, 2, -3, -4] -> kỳ vọng 0
nums = [-3, 3] -> kỳ vọng 1
nums = [1, -1, 1, -1] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Vị trí là tổng các bước đã đi

Chọn biên là tọa độ `0`. Sau mỗi đoạn ta **cộng** bước đi (có dấu) vào vị trí: `pos += nums[i]`. Số âm làm `pos` giảm (đi trái), số dương làm `pos` tăng (đi phải). Kiến "ở biên" nghĩa là **`pos == 0`**.

In [2]:
pos = 0
for x in [2, 3, -5]:
    pos += x
    print(f"đi {x:+d} -> vị trí {pos} {'(Ở BIÊN)' if pos == 0 else ''}")

đi +2 -> vị trí 2 
đi +3 -> vị trí 5 
đi -5 -> vị trí 0 (Ở BIÊN)


### 3.2. Tổng tiền tố (prefix sum)

**Tổng tiền tố** `p[i] = nums[0] + nums[1] + ... + nums[i]` chính là vị trí của kiến sau đoạn đi thứ `i`. Tính liên tục bằng vòng lặp hoặc `itertools.accumulate(nums)` (trả về iterator các tổng cộng dồn).

In [3]:
print(list(accumulate([2, 3, -5])))               # [2, 5, 0]
print(list(accumulate([3, 2, -3, -4])))           # [3, 5, 2, -2]

[2, 5, 0]
[3, 5, 2, -2]


### 3.3. Hai khái niệm: "đi ngang qua" và "dừng ở"

Khi đi từ vị trí `2` sang `-2` (đoạn `-4`), kiến đi qua các điểm `1, 0, -1, -2`: nó **đi qua** `0` nhưng **dừng ở** `-2`. Đề chỉ đếm khi **dừng** (kết thúc đoạn) ở `0`. Vì vậy ta chỉ cần xét các vị trí **cuối mỗi đoạn**, tức các tổng tiền tố, và không cần xét điểm trung gian.

In [4]:
start, move = 2, -4
path = [start + (move // abs(move)) * k for k in range(0, abs(move) + 1)]
print("các điểm kiến đi qua:", path)               # [2, 1, 0, -1, -2]
print("điểm dừng:", path[-1], "| có đi qua 0:", 0 in path[1:-1], "| dừng ở 0:", path[-1] == 0)

các điểm kiến đi qua: [2, 1, 0, -1, -2]
điểm dừng: -2 | có đi qua 0: True | dừng ở 0: False


### 3.4. `list.count` và `Counter`

`a.count(0)` đếm số phần tử bằng `0`. `Counter(a)[0]` cũng vậy và còn cho tần suất của mọi giá trị khác. Cả hai đều là cách đếm phần tử bằng một giá trị.

In [5]:
a = [2, 0, 5, 0, -1]
print(a.count(0))                                  # 2
print(Counter(a)[0], Counter(a)[7])                # 2 0

2
2 0


### 3.5. `sum(điều_kiện ...)` đếm số lần đúng

`sum(p == 0 for p in prefixes)` cộng các giá trị `True`/`False` (`True` là `1`), nên đếm được số vị trí bằng `0` mà không cần danh sách.

In [6]:
prefixes = [2, 5, 0, 3, 0]
print(sum(p == 0 for p in prefixes))               # 2

2


### 3.6. Dấu của một số và "đổi dấu"

`x > 0` là dương, `x < 0` là âm, `x == 0` đúng bằng không. Hai số **đổi dấu** (một dương một âm) **không** có nghĩa là một trong hai bằng `0`: `2` rồi `-2` đổi dấu nhưng không số nào bằng `0`. Đừng nhầm "đi qua biên" với "đứng trên biên".

In [7]:
for a, b in [(2, -2), (2, 0), (3, 5), (-1, 1)]:
    print(f"{a:+d} -> {b:+d}: đổi dấu? {(a < 0 < b) or (b < 0 < a)} | đứng ở biên (b == 0)? {b == 0}")

+2 -> -2: đổi dấu? True | đứng ở biên (b == 0)? False
+2 -> +0: đổi dấu? False | đứng ở biên (b == 0)? True
+3 -> +5: đổi dấu? False | đứng ở biên (b == 0)? False
-1 -> +1: đổi dấu? True | đứng ở biên (b == 0)? False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ta chỉ cần biết **vị trí kiến sau mỗi đoạn đi** và đếm các vị trí bằng `0`. Với `nums = [3, 2, -3, -4]`:

| đoạn `i` | bước `nums[i]` | vị trí trước | vị trí sau | ở biên? |
|---|---|---|---|---|
| 0 | `+3` | 0 | 3 | không |
| 1 | `+2` | 3 | 5 | không |
| 2 | `-3` | 5 | 2 | không |
| 3 | `-4` | 2 | -2 | không (đi **qua** biên nhưng dừng ở `-2`) |

Số lần ở biên: `0`. Với `[2, 3, -5]`: vị trí `2, 5, 0`: **có một lần** (`0`). **Vị trí xuất phát (`0`) trước khi đi không tính** (kiến "quay lại" biên, không phải "đang ở" biên lúc đầu).

In [8]:
nums = [3, 2, -3, -4]
pos, count = 0, 0
for i, x in enumerate(nums):
    before = pos
    pos += x
    at_boundary = pos == 0
    count += at_boundary
    print(f"đoạn {i}: {before:>2} {x:+d} -> {pos:>2} | ở biên: {at_boundary!s:5} | count={count}")

đoạn 0:  0 +3 ->  3 | ở biên: False | count=0
đoạn 1:  3 +2 ->  5 | ở biên: False | count=0
đoạn 2:  5 -3 ->  2 | ở biên: False | count=0
đoạn 3:  2 -4 -> -2 | ở biên: False | count=0


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng bước một

**Ý tưởng.** Mô phỏng thật đúng đề: với mỗi `nums[i]`, đi **từng bước một** `|nums[i]|` lần (mỗi lần `+1` hoặc `-1`), **không** kiểm tra giữa chừng; chỉ sau khi đi xong cả đoạn mới kiểm tra `pos == 0`.

In [9]:
def return_count_step(nums: List[int]) -> int:
    pos = 0
    count = 0
    for x in nums:
        direction = 1 if x > 0 else -1                 # +1 đi phải, -1 đi trái
        for _ in range(abs(x)):                        # đi từng đơn vị một
            pos += direction                            # (KHÔNG kiểm tra pos == 0 ở đây)
        if pos == 0:                                    # chỉ kiểm tra sau khi đi xong đoạn
            count += 1
    return count


for nums, expected in examples:
    print(nums, "->", return_count_step(nums), "(kỳ vọng", expected, ")")

[2, 3, -5] -> 1 (kỳ vọng 1 )
[3, 2, -3, -4] -> 0 (kỳ vọng 0 )
[-3, 3] -> 1 (kỳ vọng 1 )
[1, -1, 1, -1] -> 2 (kỳ vọng 2 )


In [10]:
# Xem điểm kiến đi qua và điểm dừng, ví dụ 2: [3, 2, -3, -4]
pos = 0
for x in [3, 2, -3, -4]:
    d = 1 if x > 0 else -1
    visited = []
    for _ in range(abs(x)):
        pos += d
        visited.append(pos)
    print(f"đoạn {x:+d}: đi qua {visited} | dừng ở {pos} | đi ngang qua biên: {0 in visited[:-1]} | dừng ở biên: {pos == 0}")

đoạn +3: đi qua [1, 2, 3] | dừng ở 3 | đi ngang qua biên: False | dừng ở biên: False
đoạn +2: đi qua [4, 5] | dừng ở 5 | đi ngang qua biên: False | dừng ở biên: False
đoạn -3: đi qua [4, 3, 2] | dừng ở 2 | đi ngang qua biên: False | dừng ở biên: False
đoạn -4: đi qua [1, 0, -1, -2] | dừng ở -2 | đi ngang qua biên: True | dừng ở biên: False


**Độ phức tạp (Cách 1).** Mỗi đoạn đi `|nums[i]| <= 10` bước, tổng `O(Σ|nums[i]|) <= 10 · n`: **Time `O(n · M)`** với `M = 10` (hệ số hằng nhỏ), **Space `O(1)`**.

**Nút thắt.** Đi từng đơn vị là thừa: vị trí sau một đoạn chỉ cần **cộng thẳng** `nums[i]` vào `pos`, vì đoạn đi không ảnh hưởng đến thứ gì khác ngoài điểm dừng.

## 6. Ý tưởng thuật toán / bước đột phá

Vì chỉ **điểm dừng** sau mỗi đoạn đi mới quan trọng (mục 3.3), và điểm dừng chính là **tổng tiền tố**:

> `pos_i = nums[0] + nums[1] + ... + nums[i]`

nên đáp án là **số chỉ số `i` mà tổng tiền tố bằng `0`**. Không cần mô phỏng từng bước nhỏ: một biến `pos` cộng dồn là đủ, và có thể viết gọn bằng `accumulate`. Mô hình hóa này (vị trí = tổng tiền tố, đếm tổng tiền tố bằng một giá trị) lặp lại trong rất nhiều bài về **tổng mảng con**.

## 7. Các cách tiếp theo

### Cách 2 — Cộng dồn một biến `pos`

**Giải thích.** `pos = 0`; với mỗi `x`: `pos += x`; nếu `pos == 0` thì `count += 1`.

In [11]:
def return_count_running(nums: List[int]) -> int:
    pos = 0
    count = 0
    for x in nums:
        pos += x                                       # đi trọn đoạn: cộng thẳng bước đi có dấu
        if pos == 0:                                   # dừng đúng biên
            count += 1
    return count


for nums, expected in examples:
    print(nums, "->", return_count_running(nums), "(kỳ vọng", expected, ")")

[2, 3, -5] -> 1 (kỳ vọng 1 )
[3, 2, -3, -4] -> 0 (kỳ vọng 0 )
[-3, 3] -> 1 (kỳ vọng 1 )
[1, -1, 1, -1] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — `itertools.accumulate`

**Giải thích.** `accumulate(nums)` sinh dãy tổng tiền tố một cách lười (generator). Đếm số lần bằng `0`: `sum(p == 0 for p in accumulate(nums))`.

In [12]:
def return_count_accumulate(nums: List[int]) -> int:
    return sum(p == 0 for p in accumulate(nums))        # đếm tổng tiền tố bằng 0


for nums, expected in examples:
    print(nums, "->", return_count_accumulate(nums), "(kỳ vọng", expected, ")")

print(list(accumulate([1, -1, 1, -1])))                 # [1, 0, 1, 0] -> hai số 0

[2, 3, -5] -> 1 (kỳ vọng 1 )
[3, 2, -3, -4] -> 0 (kỳ vọng 0 )
[-3, 3] -> 1 (kỳ vọng 1 )
[1, -1, 1, -1] -> 2 (kỳ vọng 2 )
[1, 0, 1, 0]


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — `Counter` của các tổng tiền tố (và liên hệ với "mảng con có tổng `0`")

**Giải thích.** `Counter(accumulate(nums))[0]` đếm số tổng tiền tố bằng `0`: chính là đáp án. Điểm đáng chú ý: `Counter` của **mọi** tổng tiền tố giúp trả lời nhiều câu hỏi khác về cùng dãy. Ví dụ, **số mảng con (liên tiếp, không rỗng) có tổng `0`** bằng số cặp `(i < j)` có `p[i] = p[j]` nếu ta thêm **tổng rỗng `0`** vào đầu dãy tiền tố. Mỗi giá trị xuất hiện `c` lần cho `c(c − 1)/2` cặp.

In [13]:
def return_count_counter(nums: List[int]) -> int:
    return Counter(accumulate(nums))[0]                 # số lần tổng tiền tố bằng 0


for nums, expected in examples:
    print(nums, "->", return_count_counter(nums), "(kỳ vọng", expected, ")")

def zero_sum_subarrays(nums):
    prefixes = [0] + list(accumulate(nums))             # thêm tổng rỗng 0 ở đầu
    return sum(c * (c - 1) // 2 for c in Counter(prefixes).values())

print("số mảng con có tổng 0 của [1, -1, 1, -1]:", zero_sum_subarrays([1, -1, 1, -1]))   # 4: [1,-1],[−1,1],[1,−1],[1,−1,1,−1]

[2, 3, -5] -> 1 (kỳ vọng 1 )
[3, 2, -3, -4] -> 0 (kỳ vọng 0 )
[-3, 3] -> 1 (kỳ vọng 1 )
[1, -1, 1, -1] -> 2 (kỳ vọng 2 )
số mảng con có tổng 0 của [1, -1, 1, -1]: 4


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (lưu mọi tổng tiền tố trong `Counter`).

In [14]:
# Code nộp LeetCode cho Cách 2 (đơn giản nhất, O(1) bộ nhớ)
from typing import List

class Solution:
    def returnToBoundaryCount(self, nums: List[int]) -> int:
        pos = 0
        count = 0
        for x in nums:
            pos += x
            if pos == 0:
                count += 1
        return count

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 với `[2, 3, -5, 4, -4, 1]` và (b) Cách 4 với bảng tổng tiền tố.

In [15]:
def trace_running(nums):
    pos, count = 0, 0
    print("nums =", nums)
    for i, x in enumerate(nums):
        before = pos
        pos += x
        hit = pos == 0
        if hit:
            count += 1
        print(f"i={i}: pos {before:>2} {x:+d} = {pos:>2} | {'VỀ BIÊN' if hit else '-':7} | count={count}")
    print("=> kết quả:", count)
    return count


assert trace_running([2, 3, -5, 4, -4, 1]) == 2
assert trace_running([3, 2, -3, -4]) == 0

nums = [2, 3, -5, 4, -4, 1]
i=0: pos  0 +2 =  2 | -       | count=0
i=1: pos  2 +3 =  5 | -       | count=0
i=2: pos  5 -5 =  0 | VỀ BIÊN | count=1
i=3: pos  0 +4 =  4 | -       | count=1
i=4: pos  4 -4 =  0 | VỀ BIÊN | count=2
i=5: pos  0 +1 =  1 | -       | count=2
=> kết quả: 2
nums = [3, 2, -3, -4]
i=0: pos  0 +3 =  3 | -       | count=0
i=1: pos  3 +2 =  5 | -       | count=0
i=2: pos  5 -3 =  2 | -       | count=0
i=3: pos  2 -4 = -2 | -       | count=0
=> kết quả: 0


In [16]:
def trace_counter(nums):
    prefixes = list(accumulate(nums))
    print("nums     =", nums)
    print("tiền tố  =", prefixes)
    c = Counter(prefixes)
    print("Counter  =", dict(c))
    print("số tổng tiền tố bằng 0:", c[0])
    return c[0]


assert trace_counter([1, -1, 1, -1]) == 2
assert trace_counter([2, 3, -5, 4, -4, 1]) == 2

nums     = [1, -1, 1, -1]
tiền tố  = [1, 0, 1, 0]
Counter  = {1: 2, 0: 2}
số tổng tiền tố bằng 0: 2
nums     = [2, 3, -5, 4, -4, 1]
tiền tố  = [2, 5, 0, 4, 0, 1]
Counter  = {2: 1, 5: 1, 0: 2, 4: 1, 1: 1}
số tổng tiền tố bằng 0: 2


Bảng tóm tắt (Cách 2, `[2, 3, -5, 4, -4, 1]`):

| `i` | `x` | `pos` | về biên? | `count` |
|---|---|---|---|---|
| 0 | +2 | 2 | không | 0 |
| 1 | +3 | 5 | không | 0 |
| 2 | −5 | 0 | **có** | 1 |
| 3 | +4 | 4 | không | 1 |
| 4 | −4 | 0 | **có** | 2 |
| 5 | +1 | 1 | không | 2 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (cộng dồn `pos`). Đúng class `Solution`, đúng method `returnToBoundaryCount(self, nums: List[int]) -> int`.

In [17]:
from typing import List

class Solution:
    def returnToBoundaryCount(self, nums: List[int]) -> int:
        pos = 0                                        # kiến xuất phát ở biên (tọa độ 0)
        count = 0                                      # số lần dừng đúng biên
        for x in nums:
            pos += x                                   # đi trọn đoạn x (dương: phải, âm: trái)
            if pos == 0:                               # chỉ xét điểm DỪNG sau mỗi đoạn
                count += 1
        return count


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.returnToBoundaryCount(nums))

[2, 3, -5] -> 1
[3, 2, -3, -4] -> 0
[-3, 3] -> 1
[1, -1, 1, -1] -> 2


## 10. Vì sao thuật toán đúng

- **Vị trí sau đoạn `i` là tổng tiền tố.** Kiến đi trái `|x|` khi `x < 0` làm vị trí giảm `|x|`, tức cộng `x`; đi phải `x` khi `x > 0` làm vị trí tăng `x`. Cả hai trường hợp chính là `pos += x`. Nên sau đoạn `i`, `pos = nums[0] + ... + nums[i]`.
- **Chỉ cần xét điểm dừng.** Đề nói chỉ kiểm tra sau khi đi **xong** `|x|` đơn vị; các điểm trung gian bỏ qua. Mỗi đoạn có đúng một điểm dừng, ứng với đúng một tổng tiền tố, nên ta đếm tổng tiền tố bằng `0` một lần cho mỗi đoạn, không sót, không thừa.
- **Không tính vị trí xuất phát.** Kiến bắt đầu ở biên nhưng "chưa quay lại" lần nào; ta chỉ kiểm tra **sau** mỗi đoạn đi.
- **Cách 1 tương đương.** Đi từng bước vẫn cho cùng điểm dừng (`+1` hoặc `-1` lặp `|x|` lần bằng `+x`), chỉ chậm hơn.
- **`accumulate` và `Counter`.** Tính cùng dãy tổng tiền tố bằng thư viện, nên cho cùng kết quả.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `M = max |nums[i]| = 10`.

- **Cách 1 (từng bước):** Time `O(n · M)`, Space `O(1)`.
- **Cách 2 (cộng dồn):** Time `O(n)`, Space `O(1)`.
- **Cách 3 (`accumulate`):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (`Counter` tiền tố):** Time `O(n)`, Space `O(n)`.
- Phải xem từng bước nên `O(n)` là tối ưu. Ở đây `n <= 100`.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "step": return_count_step,
    "running": return_count_running,
    "accumulate": return_count_accumulate,
    "counter": return_count_counter,
    "submit": sol.returnToBoundaryCount,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1], 0, "một bước: rời biên, chưa về")
check_all([-10], 0, "một bước trái dài")
check_all([5, -5], 1, "đi rồi quay lại đúng một lần")
check_all([-5, 5], 1, "đi trái rồi quay lại")
check_all([1, -1, 1, -1], 2, "về biên hai lần")
check_all([3, -3, 3, -3, 3, -3], 3, "về biên ba lần")
check_all([1, 1, -2], 1, "về biên sau hai bước tiến một bước lùi")
check_all([-1, 2], 0, "đi qua biên (từ -1 lên 1) nhưng dừng ở 1: không tính")
check_all([1, -2], 0, "đi qua biên (từ 1 xuống -1) nhưng dừng ở -1: không tính")
check_all([2, -1, -1], 1, "về biên ở bước cuối")
check_all([2, -1, -1, 3, -3], 2, "về biên hai lần")
check_all([10] * 10 + [-10] * 10, 1, "n = 20: lên 100 rồi về đúng 0 ở bước cuối")
check_all([1, -1] * 50, 50, "n = 100: xen kẽ +1, -1: về biên 50 lần")

PASS một bước: rời biên, chưa về -> 0
PASS một bước trái dài -> 0
PASS đi rồi quay lại đúng một lần -> 1
PASS đi trái rồi quay lại -> 1
PASS về biên hai lần -> 2
PASS về biên ba lần -> 3
PASS về biên sau hai bước tiến một bước lùi -> 1
PASS đi qua biên (từ -1 lên 1) nhưng dừng ở 1: không tính -> 0
PASS đi qua biên (từ 1 xuống -1) nhưng dừng ở -1: không tính -> 0
PASS về biên ở bước cuối -> 1
PASS về biên hai lần -> 2
PASS n = 20: lên 100 rồi về đúng 0 ở bước cuối -> 1
PASS n = 100: xen kẽ +1, -1: về biên 50 lần -> 50


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(11)
# Tính chất 1: số lần về biên = số tổng tiền tố bằng 0, và <= n
for _ in range(500):
    nums = [random.choice([x for x in range(-10, 11) if x != 0]) for _ in range(random.randint(1, 100))]
    r = sol.returnToBoundaryCount(nums)
    assert r == list(accumulate(nums)).count(0) and 0 <= r <= len(nums)
print("PASS đáp án = số tổng tiền tố bằng 0, nằm trong [0, n]")

# Tính chất 2: nếu tổng cả mảng bằng 0 thì đáp án >= 1 (bước cuối về biên)
for _ in range(500):
    half = [random.choice([x for x in range(-10, 11) if x != 0]) for _ in range(random.randint(1, 20))]
    nums = half + [-sum(half)] if sum(half) != 0 and abs(sum(half)) <= 10 else half
    if sum(nums) == 0:
        assert sol.returnToBoundaryCount(nums) >= 1
print("PASS tổng bằng 0 thì ít nhất một lần về biên")

# Tính chất 3: đảo dấu mọi bước (phản xạ) không đổi đáp án
for _ in range(300):
    nums = [random.choice([x for x in range(-10, 11) if x != 0]) for _ in range(random.randint(1, 50))]
    assert sol.returnToBoundaryCount(nums) == sol.returnToBoundaryCount([-x for x in nums])
print("PASS phản xạ qua biên không đổi đáp án")

# Tính chất 4: thứ tự quan trọng (hoán vị có thể đổi kết quả)
assert sol.returnToBoundaryCount([1, -1, 2, -2]) == 2 and sol.returnToBoundaryCount([1, 2, -1, -2]) == 1
print("PASS thứ tự các bước ảnh hưởng đáp án")

# Tính chất 5: đầu vào không bị sửa
nums = [2, 3, -5]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: đếm cả lúc đi ngang qua biên giữa chừng
def sai_giua_chung(nums):
    pos, count = 0, 0
    for x in nums:
        d = 1 if x > 0 else -1
        for _ in range(abs(x)):
            pos += d
            if pos == 0:                               # kiểm tra từng bước: đếm cả lúc chạm giữa chừng
                count += 1
    return count

assert sai_giua_chung([-1, 2]) == 1 and sol.returnToBoundaryCount([-1, 2]) == 0
print("PASS phản ví dụ: [-1, 2] cho", sai_giua_chung([-1, 2]), "thay vì 0")

# Phản ví dụ 2: đếm vị trí xuất phát
def sai_dem_xuat_phat(nums):
    return 1 + sol.returnToBoundaryCount(nums)

assert sai_dem_xuat_phat([3, 2, -3, -4]) == 1 != 0
print("PASS phản ví dụ: tính cả vị trí xuất phát cho", sai_dem_xuat_phat([3, 2, -3, -4]), "thay vì 0")

PASS đáp án = số tổng tiền tố bằng 0, nằm trong [0, n]
PASS tổng bằng 0 thì ít nhất một lần về biên
PASS phản xạ qua biên không đổi đáp án
PASS thứ tự các bước ảnh hưởng đáp án
PASS đầu vào không bị sửa
PASS phản ví dụ: [-1, 2] cho 1 thay vì 0
PASS phản ví dụ: tính cả vị trí xuất phát cho 1 thay vì 0


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(3028)
nonzero = [x for x in range(-10, 11) if x != 0]
for trial in range(3000):
    n = random.randint(1, 100)
    mode = random.choice(["wide", "small", "return"])
    if mode == "wide":
        nums = [random.choice(nonzero) for _ in range(n)]
    elif mode == "small":
        nums = [random.choice([-2, -1, 1, 2]) for _ in range(n)]
    else:
        nums, pos = [], 0
        for _ in range(n):
            if pos != 0 and abs(pos) <= 10 and random.random() < 0.5:
                nums.append(-pos)                       # quay thẳng về biên
                pos = 0
            else:
                x = random.choice(nonzero)
                nums.append(x)
                pos += x
    ref = return_count_step(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên (có chủ ý tạo các lần về biên) - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên (có chủ ý tạo các lần về biên) - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm cả lúc đi ngang qua biên giữa chừng

In [22]:
def sai_ngang_qua(nums):
    pos, count = 0, 0
    for x in nums:
        step = 1 if x > 0 else -1
        for _ in range(abs(x)):
            pos += step
            if pos == 0:                                # kiểm tra sau MỖI đơn vị
                count += 1
    return count


print("sai  :", sai_ngang_qua([-1, 2]), "| đúng:", Solution().returnToBoundaryCount([-1, 2]))

sai  : 1 | đúng: 0


**Vì sao sai:** với `[-1, 2]` kiến đi từ `-1` lên `1` và **đi ngang qua** `0` nhưng **dừng ở `1`**. Đề chỉ đếm khi kiến **đứng ở biên sau khi hết một đoạn đi**. Hãy chỉ kiểm tra `pos == 0` **sau khi** cộng cả `nums[i]` vào `pos`.

### Lỗi 2: Tính cả vị trí xuất phát

In [23]:
def sai_xuat_phat(nums):
    pos = 0
    count = 1 if pos == 0 else 0                        # tính luôn vị trí ban đầu
    for x in nums:
        pos += x
        if pos == 0:
            count += 1
    return count


print("sai  :", sai_xuat_phat([3, 2, -3, -4]), "| đúng:", Solution().returnToBoundaryCount([3, 2, -3, -4]))

sai  : 1 | đúng: 0


**Vì sao sai:** kiến **xuất phát** ở biên, nhưng đề đếm số lần nó **quay lại** biên **sau khi đi**. Lúc chưa đi bước nào không phải một lần "quay lại". Chỉ kiểm tra sau mỗi lần cộng.

### Lỗi 3: Nhầm "đổi dấu" với "chạm biên"

In [24]:
def sai_doi_dau(nums):
    pos, count = 0, 0
    for x in nums:
        new = pos + x
        if (pos < 0 < new) or (new < 0 < pos) or new == 0:     # đếm cả khi vượt qua biên
            count += 1
        pos = new
    return count


print("sai  :", sai_doi_dau([-1, 2, -3]), "| đúng:", Solution().returnToBoundaryCount([-1, 2, -3]))

sai  : 2 | đúng: 0


**Vì sao sai:** hai vị trí liên tiếp **trái dấu** (`-1` rồi `1`) chỉ cho biết kiến **đi qua** biên; kiến không đứng ở đó. Với `[-1, 2, -3]`: vị trí `-1, 1, -2`: đổi dấu hai lần nhưng không lần nào `== 0`; đáp án đúng `0`.

### Lỗi 4: Gán thay vì cộng dồn

In [25]:
def sai_gan(nums):
    count = 0
    for x in nums:
        pos = x                                         # quên cộng dồn: mất vị trí trước đó
        if pos == 0:
            count += 1
    return count


print("sai  :", sai_gan([2, 3, -5]), "| đúng:", Solution().returnToBoundaryCount([2, 3, -5]))

sai  :

 0 | đúng: 1


**Vì sao sai:** vị trí **phụ thuộc mọi bước trước**, nên phải `pos += x`. `pos = x` coi mỗi bước như bắt đầu lại từ biên và (vì `nums[i] != 0`) không bao giờ bằng `0`, cho đáp án `0` thay vì `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "vị trí sau mỗi bước, bắt đầu từ `0`" → **tổng tiền tố**; `pos += x`.
- Công thức nhớ: **đáp án = số tổng tiền tố bằng `0`** = `sum(p == 0 for p in accumulate(nums))`.
- Đọc kỹ "chỉ kiểm tra **sau khi** đi xong": nghĩa là **không** xét các điểm trung gian.
- Phân biệt "**đi qua** `0`" (đổi dấu) với "**dừng ở** `0`" (`== 0`).
- `Counter` các tổng tiền tố giúp đếm **mảng con có tổng cho trước**: thêm tổng rỗng `0` đầu dãy, cặp tiền tố bằng nhau cho một mảng con tổng `0`.
- Khi mô phỏng nhỏ nên viết luôn bản "từng bước" để làm chuẩn so sánh cho bản tối ưu.

## 15. Biến thể của bài

1. Đếm số lần kiến **đi ngang qua hoặc dừng ở** biên (mọi lần chạm điểm `0`, kể cả giữa chừng).
2. Trả về **vị trí cuối cùng** của kiến.
3. Trả về **khoảng cách xa nhất** từ biên mà kiến từng đến (sau mỗi bước, hoặc kể cả giữa chừng).
4. Đếm số lần kiến **đổi phía** (từ phía dương sang phía âm hoặc ngược lại).
5. Đếm số **mảng con liên tiếp** có tổng `0` (liên hệ Cách 4).

Thử biến thể 1, 2, 3 và 5:

In [26]:
def touches_including_midway(nums):
    pos, count = 0, 0
    for x in nums:
        d = 1 if x > 0 else -1
        for _ in range(abs(x)):
            pos += d
            count += pos == 0
    return count


def final_position(nums):
    return sum(nums)


def farthest_after_steps(nums):
    return max(abs(p) for p in accumulate(nums))


print("1.", touches_including_midway([-1, 2]), touches_including_midway([3, 2, -3, -4]))   # 1 1
print("2.", final_position([2, 3, -5, 4]))                                                    # 4
print("3.", farthest_after_steps([3, 2, -3, -4]))                                             # 5
print("5.", zero_sum_subarrays([2, 3, -5, 4, -4, 1]))                                         # 3
assert touches_including_midway([-1, 2]) == 1 and touches_including_midway([3, 2, -3, -4]) == 1
assert final_position([2, 3, -5, 4]) == 4 and farthest_after_steps([3, 2, -3, -4]) == 5
assert zero_sum_subarrays([2, 3, -5, 4, -4, 1]) == 3

1. 1 1
2. 4
3. 5
5. 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng bước một | đi `+1`/`-1`, kiểm tra cuối đoạn | `O(n·M)` | `O(1)` | hiểu đúng đề, chuẩn so sánh |
| 2. Cộng dồn `pos` | `pos += x`, kiểm tra `pos == 0` | `O(n)` | `O(1)` | **cách nộp bài**, đơn giản nhất |
| 3. `accumulate` | đếm tổng tiền tố bằng `0` | `O(n)` | `O(1)` | code gọn |
| 4. `Counter` tiền tố | tần suất mọi tổng tiền tố | `O(n)` | `O(n)` | liên hệ bài mảng con có tổng cho trước |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(accumulate([1, 2, -3]))`, `[0, 5, 0].count(0)`, `Solution().returnToBoundaryCount([4, -4])`, `Solution().returnToBoundaryCount([2, -3, 1])`, `Solution().returnToBoundaryCount([-2, -3, 5])`.

**Bài 2 (tính tay).** Với `nums = [4, -1, -3, 5, -2, -3]`: ghi vị trí sau từng bước, đánh dấu lần về biên, và cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành `return_count_todo(nums)` theo Cách 2.

In [27]:
def return_count_todo(nums):
    pos = 0
    count = 0
    for x in nums:
        # TODO 1: cộng bước đi x vào pos
        # TODO 2: nếu pos == 0 thì tăng count lên 1
        pass
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `return_count_accumulate2(nums)` bằng **`accumulate`** (Cách 3) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `side_changes(nums)`: số lần kiến **chuyển phía** (từ phía dương sang phía âm hoặc ngược lại, xét các điểm dừng; điểm dừng ở `0` không thuộc phía nào, hãy bỏ qua nó khi xác định phía). (b) Viết `first_return_index(nums)`: chỉ số `i` của **lần đầu tiên** về biên, hoặc `-1` nếu không bao giờ về. (c) Viết `max_distance_midway(nums)`: khoảng cách **xa nhất** từ biên kể cả giữa chừng (vì kiến đi trên đường thẳng, đó chính là `max |pos|` ở các điểm dừng; hãy giải thích vì sao).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 3, 0]`, `2`, `1` (`[4, -4]`: vị trí `4, 0`), `1` (`[2, -3, 1]`: vị trí `2, -1, 0`), `1` (`[-2, -3, 5]`: vị trí `-2, -5, 0`). Bài 3: `pos += x` rồi `if pos == 0: count += 1`.
- **Mức 2 (vừa):** Bài 2: vị trí `4, 3, 0, 5, 3, 0`: về biên ở `i = 2` và `i = 5`: đáp án `2`. Bài 4: `sum(p == 0 for p in accumulate(nums))`.
- **Mức 3 (sâu):** Bài 5(a): duyệt các điểm dừng, giữ `side` (dấu của điểm dừng khác `0` gần nhất), đếm khi `side` đổi. 5(b): `next((i for i, p in enumerate(accumulate(nums)) if p == 0), -1)`. 5(c): giữa hai điểm dừng liên tiếp, kiến đi thẳng một mạch nên khoảng cách tới biên là đơn điệu rồi giảm/tăng, cực đại `|pos|` chỉ có thể ở một **đầu mút** (điểm dừng) hoặc ở biên (giá trị `0`), nên `max |pos|` trên các điểm dừng là đủ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert list(accumulate([1, 2, -3])) == [1, 3, 0] and [0, 5, 0].count(0) == 2
assert Solution().returnToBoundaryCount([4, -4]) == 1
assert Solution().returnToBoundaryCount([2, -3, 1]) == 1
assert Solution().returnToBoundaryCount([-2, -3, 5]) == 1

# Bài 2
assert list(accumulate([4, -1, -3, 5, -2, -3])) == [4, 3, 0, 5, 3, 0]
assert Solution().returnToBoundaryCount([4, -1, -3, 5, -2, -3]) == 2

# Bài 3
def return_count_todo_ans(nums):
    pos = 0
    count = 0
    for x in nums:
        pos += x
        if pos == 0:
            count += 1
    return count

for nums, expected in examples:
    assert return_count_todo_ans(nums) == expected

# Bài 4
def return_count_accumulate2(nums):
    return sum(p == 0 for p in accumulate(nums))

for nums, expected in examples:
    assert return_count_accumulate2(nums) == expected

# Bài 5(a)
def side_changes(nums):
    side, changes = 0, 0
    for p in accumulate(nums):
        if p == 0:
            continue
        s = 1 if p > 0 else -1
        if side != 0 and s != side:
            changes += 1
        side = s
    return changes

assert side_changes([-1, 2, -3]) == 2
assert side_changes([3, -3, 3, -3]) == 0
assert side_changes([2, -4, 4]) == 2

# Bài 5(b)
def first_return_index(nums):
    return next((i for i, p in enumerate(accumulate(nums)) if p == 0), -1)

assert first_return_index([2, 3, -5, 1]) == 2
assert first_return_index([1, 1]) == -1

# Bài 5(c)
def max_distance_midway(nums):
    return max(abs(p) for p in accumulate(nums))

def max_distance_by_steps(nums):
    pos, best = 0, 0
    for x in nums:
        d = 1 if x > 0 else -1
        for _ in range(abs(x)):
            pos += d
            best = max(best, abs(pos))
    return best

for _ in range(200):
    arr = [random.choice(nonzero) for _ in range(random.randint(1, 30))]
    assert max_distance_midway(arr) == max_distance_by_steps(arr)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm tổng tiền tố bằng `0`)

```python
from typing import List

class Solution:
    def returnToBoundaryCount(self, nums: List[int]) -> int:
        pos = 0
        count = 0
        for x in nums:
            pos += x
            if pos == 0:
                count += 1
        return count
```

**Checklist**

- [ ] Cộng dồn `pos += x` (không `pos = x`)?
- [ ] Chỉ kiểm tra `pos == 0` **sau khi** đi xong cả đoạn (không đếm lúc đi ngang qua)?
- [ ] Không tính vị trí xuất phát lúc chưa đi?
- [ ] Phân biệt "đổi dấu" (đi qua biên) với "bằng `0`" (dừng ở biên)?
- [ ] Đã thử: một bước, đi rồi quay về (`[5, -5]`), đi ngang qua biên (`[-1, 2]`), xen kẽ `+1, -1`, `n = 100`?
- [ ] Skeleton đúng `class Solution` / `def returnToBoundaryCount(self, nums: List[int]) -> int`?